<h1>2. Tools and Sequential Crews</h1>

In [1]:
import warnings

# Suppress Pydantic V2 warnings BEFORE importing CrewAI
warnings.filterwarnings('ignore', category=DeprecationWarning, module='pydantic.*')
warnings.filterwarnings('ignore', category=UserWarning, module='pydantic.*')

from collections import Counter
from pprint import pprint

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 

import os
import litellm
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool, ScrapeWebsiteTool
from langchain.tools import Tool
from crewai_tools import tool

# Configure LiteLLM to handle Anthropic API message format requirements
litellm.modify_params = True

load_dotenv()

import watermark

%load_ext watermark
%matplotlib inline

/Users/bgoncalves/My Drive/DataForScience/CrewAI/.venv/lib/python3.12/site-packages/pydantic/_internal/_config.py:295: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)
/Users/bgoncalves/My Drive/DataForScience/CrewAI/.venv/lib/python3.12/site-packages/IPython/core/pylabtools.py:77: DeprecationWarning: backend2gui is deprecated since IPython 8.24, backends are managed in matplotlib and can be externally registered.
  warnings.warn(


We start by printing out the versions of the libraries we're using for future reference

In [2]:
%watermark -n -v -m -g -iv

Python implementation: CPython
Python version       : 3.12.10
IPython version      : 9.10.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 25.2.0
Machine     : arm64
Processor   : arm
CPU cores   : 16
Architecture: 64bit

Git hash: 2b98c2c19a22d9b5e74349315b8e0b33a6a32a32

matplotlib    : 3.8.4
litellm       : 1.60.0
pandas        : 2.2.3
crewai        : 0.86.0
langchain_core: 0.3.63
watermark     : 2.5.0
crewai_tools  : 0.17.0
numpy         : 1.26.4



Load default figure style

In [3]:
plt.style.use('d4sci.mplstyle')

# 2. Tools and Sequential Crews

In this notebook, we'll learn how to:
- Equip agents with **tools** (search, APIs, custom functions)
- Create **custom tools** for specific needs
- Organize agents into **sequential workflows**
- Understand **context passing** between agents

Tools are what make agents truly powerful - they can search the web, call APIs, read files, and execute custom functions.

## Setup

In [4]:
# Use CrewAI's native LLM class with LiteLLM provider prefix
llm = LLM(model="anthropic/claude-sonnet-4-5-20250929", temperature=0.7)

## Using Pre-built Tools: Search

CrewAI provides several pre-built tools. One of the most useful is the search tool, which allows agents to search the web for current information.

In [5]:
# Initialize search tool (requires SERPER_API_KEY environment variable)
# Get your free API key at: https://serper.dev/
search_tool = SerperDevTool()

Create an agent with search capabilities

In [6]:
researcher = Agent(
    role="Market Research Analyst",
    goal="Research current market trends and gather relevant data",
    backstory="""
        You are an expert market researcher with access to web search.
        You excel at finding the most relevant and up-to-date information
        about market trends, competitors, and industry developments.
    """,
    tools=[search_tool],  # Giving the agent access to search
    llm=llm,
    verbose=True
)

Define a research task

In [7]:
research_task = Task(
    description="Research the latest trends in AI agent frameworks in 2026",
    expected_output="A summary of the top 3 AI agent frameworks with key features",
    agent=researcher # <- The agent that is able to perform this task
)

In [8]:
crew = Crew(
    agents=[researcher],
    tasks=[research_task],
    verbose=True
)

result = crew.kickoff()
print("\n" + "="*80)
print("RESEARCH RESULT:")
print("="*80)
print(result)

# Agent: Market Research Analyst
## Task: Research the latest trends in AI agent frameworks in 2026


/Users/bgoncalves/My Drive/DataForScience/CrewAI/.venv/lib/python3.12/site-packages/crewai/tools/tool_usage.py:162: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  acceptable_args = tool.args_schema.schema()["properties"].keys()  # type: ignore # Item "None" of "type[BaseModel] | None" has no attribute "schema"




# Agent: Market Research Analyst
## Thought: I need to search for the latest trends in AI agent frameworks in 2026 to provide a comprehensive summary of the top 3 frameworks with their key features.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"top AI agent frameworks 2026 latest trends\"}"
## Tool Output: 

Search results: Title: 12 Best AI Agent Frameworks in 2026 | Data Science Collective
Link: https://medium.com/data-science-collective/the-best-ai-agent-frameworks-for-2026-tier-list-b3a4362fac0d
Snippet: The Best AI Agent Frameworks for 2026 · 1 — LangGraph · 2 — CrewAI · 5 — Semantic Kernel · 6 — Pydantic AI · 7 — Claude MCP (Model Context Protocol).
---
Title: A Detailed Comparison of Top 6 AI Agent Frameworks in 2026 - Turing
Link: https://www.turing.com/resources/ai-agent-frameworks
Snippet: This guide compares six leading AI agent frameworks—LangGraph, LlamaIndex, CrewAI, Microsoft Semantic Kernel, Microsoft AutoGen, and OpenAI ...
---
Title: Top 5

/Users/bgoncalves/My Drive/DataForScience/CrewAI/.venv/lib/python3.12/site-packages/crewai/tools/tool_usage.py:162: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  acceptable_args = tool.args_schema.schema()["properties"].keys()  # type: ignore # Item "None" of "type[BaseModel] | None" has no attribute "schema"




# Agent: Market Research Analyst
## Thought: Thought: I have good initial information. Let me search for more specific details about the top frameworks mentioned - LangGraph, CrewAI, and their key features.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"LangGraph CrewAI key features 2026 comparison\"}"
## Tool Output: 

Search results: Title: CrewAI vs LangGraph in 2026: What Actually Works for Real Multi ...
Link: https://medium.com/@vikrantdheer/crewai-vs-langgraph-in-2026-what-actually-works-for-real-multi-agent-systems-6c4c979212cb
Snippet: CrewAI: Role-Based Agent Collaboration · Less explicit control over branching and state · Debugging becomes harder as workflows grow · Custom ...
---
Title: AI Agent Frameworks: A Detailed Comparison
Link: https://www.turing.com/resources/ai-agent-frameworks
Snippet: Read how top AI agent frameworks, including LangGraph, LlamaIndex, CrewAI, Semantic Kernel, AutoGen, and Swarm, compare in terms of features ...
---
Titl

...



# Agent: Market Research Analyst
## Thought: Thought: I need more detailed information about the key features of these top frameworks. Let me search for specific details about LangGraph, CrewAI, and another top framework.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"LangGraph 2026 key features stateful workflows\"}"
## Tool Output: 

Search results: Title: LangGraph - Build controllable agents
Link: https://www.langchain.com/langgraph
Snippet: Build expressive, customizable agent workflows. LangGraph's low-level primitives provide the flexibility needed to create fully customizable agents. Design ...
---
Title: Building Intelligent, Stateful Workflows with LangGraph
Link: https://www.linkedin.com/pulse/building-intelligent-stateful-workflows-langgraph-practical-kohli-w4m0c
Snippet: LangGraph provides the essential building blocks to design stateful, modular, and maintainable AI workflows with clarity and precision. As ...
---
Title: Before You Upgrade to L

/Users/bgoncalves/My Drive/DataForScience/CrewAI/.venv/lib/python3.12/site-packages/crewai/tools/tool_usage.py:162: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  acceptable_args = tool.args_schema.schema()["properties"].keys()  # type: ignore # Item "None" of "type[BaseModel] | None" has no attribute "schema"




# Agent: Market Research Analyst
## Thought: Thought: Let me search for more details about CrewAI and Microsoft AutoGen/Semantic Kernel to complete the top 3 frameworks analysis.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"CrewAI 2026 features role-based collaboration multi-agent\"}"
## Tool Output: 

Search results: Title: The Leading Multi-Agent Platform
Link: https://www.crewai.com/
Snippet: CrewAI makes it easy for enterprises to operate teams of AI agents that perform complex tasks autonomously, reliably and with full control.
---
Title: crewAIInc/crewAI
Link: https://github.com/crewAIInc/crewAI
Snippet: Crews: Teams of AI agents with true autonomy and agency, working together to accomplish complex tasks through role-based collaboration. Crews enable ...
---
Title: Collaboration
Link: https://docs.crewai.com/en/concepts/collaboration
Snippet: Collaboration in CrewAI enables agents to work together as a team by delegating tasks and asking questions to

/Users/bgoncalves/My Drive/DataForScience/CrewAI/.venv/lib/python3.12/site-packages/crewai/tools/tool_usage.py:162: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  acceptable_args = tool.args_schema.schema()["properties"].keys()  # type: ignore # Item "None" of "type[BaseModel] | None" has no attribute "schema"




# Agent: Market Research Analyst
## Thought: Thought: Now let me search for Microsoft AutoGen or Semantic Kernel features to round out the top 3 frameworks.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"Microsoft AutoGen Semantic Kernel 2026 key features\"}"
## Tool Output: 

Search results: Title: Microsoft Agent Framework Overview
Link: https://learn.microsoft.com/en-us/agent-framework/overview/
Snippet: Agent Framework combines AutoGen's simple agent abstractions with Semantic Kernel's enterprise features — session-based state management ...
---
Title: Semantic Kernel + AutoGen = Open-Source 'Microsoft ...
Link: https://visualstudiomagazine.com/articles/2025/10/01/semantic-kernel-autogen--open-source-microsoft-agent-framework.aspx
Snippet: Semantic Kernel is a lightweight, open-source SDK designed to help developers build intelligent AI agents and orchestrate complex workflows ...
---
Title: Introducing Microsoft Agent Framework
Link: https://azure.micro

## Creating Custom Tools

You can create your own tools for specific needs. A tool is simply a function with a clear description.

In [12]:
# Define a custom calculator tool
@tool("Stock Price Calculator")
def calculate_stock_metrics(stock_data: str) -> str:
    """
    Calculate basic stock metrics from price data.
    Input format: 'current_price,yearly_high,yearly_low'
    Example: '150.25,180.50,120.00'
    """
    try:
        current, high, low = [float(x) for x in stock_data.split(',')]
        
        # Calculate metrics
        distance_from_high = ((high - current) / high) * 100
        distance_from_low = ((current - low) / low) * 100
        volatility_range = ((high - low) / low) * 100
        
        return f"""
        Stock Metrics Analysis:
        - Current Price: ${current:.2f}
        - 52-Week High: ${high:.2f}
        - 52-Week Low: ${low:.2f}
        - Distance from High: {distance_from_high:.1f}%
        - Gain from Low: {distance_from_low:.1f}%
        - Volatility Range: {volatility_range:.1f}%
        """
    except Exception as e:
        return f"Error calculating metrics: {str(e)}"

# Note: The tool will be used by agents in the crew
print("Custom stock calculator tool created successfully")

Custom stock calculator tool created successfully


## Sequential Crews: Multi-Agent Workflows

Real-world tasks often require multiple agents working in sequence, where each agent builds on the work of the previous one. This is the power of **sequential crews**.

In [13]:
# Create a team of financial analysts

# Agent 1: Researcher
data_researcher = Agent(
    role="Financial Data Researcher",
    goal="Gather comprehensive financial data and market information",
    backstory="""
        You are a meticulous financial researcher with expertise in finding
        and organizing financial data. You use search tools to gather the
        most current and relevant information.
    """,
    tools=[search_tool],
    llm=llm,
    verbose=True
)

# Agent 2: Analyst
financial_analyst = Agent(
    role="Senior Financial Analyst",
    goal="Analyze financial data and identify investment opportunities",
    backstory="""
        You are a senior financial analyst with 15 years of experience.
        You excel at interpreting financial data and providing actionable
        investment insights. You use quantitative tools to support your analysis.
    """,
    tools=[calculate_stock_metrics],
    llm=llm,
    verbose=True
)

# Agent 3: Report Writer
report_writer = Agent(
    role="Investment Report Writer",
    goal="Create clear, professional investment reports",
    backstory="""
        You are an expert at synthesizing complex financial analysis into
        clear, actionable investment reports. You write for institutional
        investors who need concise, well-structured recommendations.
    """,
    llm=llm,
    verbose=True
)

print("Created 3-agent financial analysis team")

Created 3-agent financial analysis team


## Understanding Context: Tasks Pass Information

In a sequential crew, each task can access the output of previous tasks through **context**. This allows agents to build on each other's work.

In [14]:
# Task 1: Research
research_task = Task(
    description="""
        Research Apple Inc. (AAPL) stock:
        1. Current stock price
        2. Recent news and developments
        3. Analyst sentiment
        4. Key financial metrics
    """,
    expected_output="Comprehensive research summary with current data",
    agent=data_researcher
)

# Task 2: Analysis (receives research_task output as context)
analysis_task = Task(
    description="""
        Based on the research provided, perform a detailed analysis:
        1. Evaluate the company's current position
        2. Identify key opportunities and risks
        3. Calculate relevant metrics using the stock calculator tool
        4. Provide an investment recommendation (Buy/Hold/Sell)
    """,
    expected_output="Detailed financial analysis with recommendation",
    agent=financial_analyst,
    context=[research_task]  # This task receives research_task output
)

# Task 3: Report Writing (receives both previous outputs as context)
report_task = Task(
    description="""
        Create a professional investment report:
        1. Executive Summary
        2. Key Findings
        3. Financial Analysis
        4. Investment Recommendation
        5. Risk Factors
        
        Format the report in a clear, structured manner suitable for
        presentation to investment committee.
    """,
    expected_output="Professional investment report in markdown format",
    agent=report_writer,
    context=[research_task, analysis_task]  # Receives both previous outputs
)

print("Defined sequential task chain with context passing")

Defined sequential task chain with context passing


## Execute the Sequential Crew

Now let's run the entire workflow. Watch how information flows from researcher → analyst → writer.

Create the crew with sequential process

In [15]:
financial_crew = Crew(
    agents=[
        data_researcher, 
        financial_analyst, 
        report_writer
    ],
    tasks=[
        research_task, 
        analysis_task, 
        report_task
    ],
    process=Process.sequential,  # Execute tasks in order
    verbose=True
)

print("\n" + "="*80)
print("EXECUTING: Financial Analysis Workflow")
print("="*80 + "\n")

2026-02-19 14:28:27,596 - 8505535552 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed



EXECUTING: Financial Analysis Workflow



Kickoff!

In [16]:
result = financial_crew.kickoff()

print("\n" + "="*80)
print("FINAL INVESTMENT REPORT:")
print("="*80)
print(result)

# Agent: Financial Data Researcher
## Task: 
        Research Apple Inc. (AAPL) stock:
        1. Current stock price
        2. Recent news and developments
        3. Analyst sentiment
        4. Key financial metrics
    


/Users/bgoncalves/My Drive/DataForScience/CrewAI/.venv/lib/python3.12/site-packages/crewai/tools/tool_usage.py:162: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  acceptable_args = tool.args_schema.schema()["properties"].keys()  # type: ignore # Item "None" of "type[BaseModel] | None" has no attribute "schema"




# Agent: Financial Data Researcher
## Thought: I'll help you research Apple Inc. (AAPL) stock comprehensively. Let me gather the current information across all the requested areas.
Thought: I need to search for current Apple stock price and financial information first.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"Apple AAPL stock price current 2024\"}"
## Tool Output: 

Search results: Title: Apple Inc. (AAPL) Stock Historical Prices & Data - Yahoo Finance
Link: https://finance.yahoo.com/quote/AAPL/history/
Snippet: Discover historical prices for AAPL stock on Yahoo Finance. View daily, weekly or monthly format back to when Apple Inc. stock was issued.
---
Title: Stock Price - Apple - Investor Relations
Link: https://investor.apple.com/stock-price/default.aspx
Snippet: Stock Quote: NASDAQ: AAPL · Day's Open263.60 · Closing Price264.35 · Volume34.2M · Intraday High266.82 · Intraday Low262.45.
---
Title: Apple - 46 Year Stock Price History | AAPL - Macrotren

/Users/bgoncalves/My Drive/DataForScience/CrewAI/.venv/lib/python3.12/site-packages/crewai/tools/tool_usage.py:162: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  acceptable_args = tool.args_schema.schema()["properties"].keys()  # type: ignore # Item "None" of "type[BaseModel] | None" has no attribute "schema"




# Agent: Financial Data Researcher
## Thought: Thought: Good, I have current stock price information showing AAPL around $264. Now let me search for recent news and developments.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"Apple AAPL recent news developments 2024 2025\"}"
## Tool Output: 

Search results: Title: Apple (NASDAQ:AAPL) Delivers Impressive Q4 CY2025
Link: https://finance.yahoo.com/news/apple-nasdaq-aapl-delivers-impressive-214030985.html
Snippet: iPhone and iPad maker Apple (NASDAQ:AAPL) reported Q4 CY2025 results beating Wall Street's revenue expectations, with sales up 15.7% year on ...
---
Title: Apple will spend more than $500 billion in the U.S. over the next four ...
Link: https://www.apple.com/newsroom/2025/02/apple-will-spend-more-than-500-billion-usd-in-the-us-over-the-next-four-years/
Snippet: Apple today announced its largest-ever spend commitment, with plans to spend and invest more than $500 billion in the US over the next four ye

...



# Agent: Financial Data Researcher
## Thought: Thought: Excellent, I have recent news including Q4 2025 results, $500 billion US investment commitment, and AI developments. Now let me search for analyst sentiment and ratings.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"Apple AAPL analyst ratings sentiment price target 2025 2026\"}"
## Tool Output: 

Search results: Title: Apple (AAPL) Stock Forecast and Price Target 2026 - MarketBeat
Link: https://www.marketbeat.com/stocks/NASDAQ/AAPL/forecast/
Snippet: According to the 36 analysts' twelve-month price targets for Apple, the average price target is $292.13. The highest price target for AAPL is $350.00, while ...
---
Title: Apple analyst sets bold stock target for 2026 - Yahoo Finance
Link: https://finance.yahoo.com/news/apple-analyst-sets-bold-stock-040700179.html
Snippet: Wedbush analyst Dan Ives raised his stock price target for Apple to $350 from $320, reiterating an outperform rating. “2026 is going to

/Users/bgoncalves/My Drive/DataForScience/CrewAI/.venv/lib/python3.12/site-packages/crewai/tools/tool_usage.py:162: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  acceptable_args = tool.args_schema.schema()["properties"].keys()  # type: ignore # Item "None" of "type[BaseModel] | None" has no attribute "schema"




# Agent: Financial Data Researcher
## Thought: Thought: I have gathered comprehensive information about Apple's stock price, recent news, and analyst sentiment. Now I need to search for key financial metrics to complete the research.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"Apple AAPL key financial metrics PE ratio market cap revenue earnings 2025\"}"
## Tool Output: 

Search results: Title: Apple Inc. ( AAPL) Key Metrics - Digrin
Link: https://www.digrin.com/stocks/detail/AAPL/key_metrics
Snippet: AAPL Revenue per Share metrics ; 2026Q1, $9.75, $2.85 ; 2025Q4, $6.85, $1.84 ; 2025Q3, $6.31, $1.57 ; 2025Q2, $6.36, $1.65 ; 2025Q1, $8.24, $2.41 ...
---
Title: Apple (AAPL) P/E Ratio: Current & Historical Analysis
Link: https://public.com/stocks/aapl/pe-ratio
Snippet: The P/E ratio for Apple (AAPL) is 33.39 as of Feb 17, 2026. This represents a decrease of -4.44% compared to its 12-month average P/E ratio of 34.94.
---
Title: Apple Inc. (AAPL) Financial Rat

/Users/bgoncalves/My Drive/DataForScience/CrewAI/.venv/lib/python3.12/site-packages/crewai/tools/tool_usage.py:162: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  acceptable_args = tool.args_schema.schema()["properties"].keys()  # type: ignore # Item "None" of "type[BaseModel] | None" has no attribute "schema"




# Agent: Senior Financial Analyst
## Thought: I need to perform a comprehensive financial analysis of Apple Inc. (AAPL) based on the research provided. Let me start by calculating key stock metrics using the available tool.
## Using tool: Stock Price Calculator
## Tool Input: 
"{\"stock_data\": \"264.35,288.62,205.00\"}"
## Tool Output: 

        Stock Metrics Analysis:
        - Current Price: $264.35
        - 52-Week High: $288.62
        - 52-Week Low: $205.00
        - Distance from High: 8.4%
        - Gain from Low: 29.0%
        - Volatility Range: 40.8%
        


# Agent: Senior Financial Analyst
## Final Answer: 
# COMPREHENSIVE FINANCIAL ANALYSIS: APPLE INC. (AAPL)
## Investment Recommendation Report - February 18, 2026

---

## EXECUTIVE SUMMARY

**Current Price:** $264.35  
**Recommendation:** **BUY**  
**Price Target (12-month):** $305-$315  
**Expected Return:** 15-19% upside potential  
**Risk Rating:** Moderate  
**Investment Horizon:** Medium to Long-term (12-24 mo